# 🚀 CS 5542 — SD API Server (A100 Edition)

Optimized for **A100 (40 GB VRAM)** — SDXL at 1024 px, bfloat16, batched generation, and notebook-friendly quality score persistence.

```
Browser (HTML app) → POST /generate → ngrok → Flask + A100 GPU → base64 images back
Browser (HTML app) → POST /quality/submit → ngrok → Flask → results/quality_scores.csv
```

**Steps:**
1. Run Cell 1 — mount Drive, install packages
2. Run Cell 2 — load SDXL pipeline and configure results paths
3. Run Cell 3 — Flask API routes for generation and quality scoring
4. Run Cell 4 — start ngrok + Flask → copy the URL into your HTML app

**Quality Review Options:**
- Use the separate `colab_quality_review.ipynb` notebook for widget-based manual review inside Colab
- Or send ratings directly from your frontend to `POST /quality/submit` and inspect `GET /quality/scores` / `GET /quality/summary`

In [ ]:
# ── CELL 1 ── Clone GitHub repo + Install ───────────────────────────────────
import subprocess, sys

# ▼▼ EDIT: your GitHub repo URL ▼▼
GITHUB_REPO = 'https://github.com/mosomo82/COMP_SCI_5542.git'
# For a private repo use a token:
# GITHUB_REPO = 'https://YOUR_TOKEN@github.com/YOUR_USERNAME/GENAI_Stable_Diffussion_Challenge'
# ▲▲─────────────────────────────▲▲

CLONE_DIR = '/content/COMP_SCI_5542/GENAI_Stable_Diffussion_Challenge'

import os
if not os.path.exists(CLONE_DIR):
    subprocess.run(['git', 'clone', GITHUB_REPO, CLONE_DIR], check=True)
    print(f'Cloned to {CLONE_DIR}')
else:
    subprocess.run(['git', '-C', CLONE_DIR, 'pull'], check=True)
    print(f'Pulled latest changes into {CLONE_DIR}')

pkgs = [
    'diffusers',
    'transformers',
    'huggingface_hub>=0.23.0',
    'accelerate', 'controlnet-aux',
    'flask', 'flask-cors', 'pyngrok', 'rich', 'xformers',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-U'] + pkgs, check=True)
print('Packages ready')

import torch
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB')
print(f'BF16: {torch.cuda.is_bf16_supported()}')

In [ ]:
# ── CELL 2 ── Setup repo path + A100 pipeline loader ────────────────────────
import sys, os, threading, warnings, logging
from pathlib import Path
import torch

# Suppress torch.compile symbolic-math noise
logging.getLogger('torch._dynamo').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=UserWarning, module='torch')

REPO_PATH = Path('/content/COMP_SCI_5542/GENAI_Stable_Diffussion_Challenge')
assert REPO_PATH.exists(), f'Repo not found: {REPO_PATH} — did Cell 1 finish?'
sys.path.insert(0, str(REPO_PATH))
print(f'Repo path: {REPO_PATH}')

RESULTS_DIR = REPO_PATH / 'results'
OUTPUTS_DIR = REPO_PATH / 'outputs'
QUALITY_SCORES_PATH = RESULTS_DIR / 'quality_scores.csv'
QUALITY_SUMMARY_PATH = RESULTS_DIR / 'quality_summary.csv'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Results dir: {RESULTS_DIR}')

from pipeline.sd_pipeline    import load_sd_pipeline, load_controlnet_pipeline
from pipeline.prompt_builder import naive_prompt, structured_prompt, NEGATIVE_PROMPT
from pipeline.evaluator      import clip_score
print('Pipeline modules imported')

DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'dtype: {DTYPE}')

MODEL_IDS = {
    'sd15' : 'runwayml/stable-diffusion-v1-5',
    'sd21' : 'stabilityai/stable-diffusion-2-1',
    'sdxl' : 'stabilityai/stable-diffusion-xl-base-1.0',
}
DEFAULTS = {
    'sd15': dict(resolution=512,  steps=30, cfg=7.5),
    'sd21': dict(resolution=768,  steps=30, cfg=7.5),
    'sdxl': dict(resolution=1024, steps=25, cfg=7.0),
}

_pipelines = {}
_lock = threading.Lock()

def get_pipeline(model_key='sdxl', use_controlnet=False):
    key = f'{model_key}_{"cn" if use_controlnet else "sd"}'
    with _lock:
        if key not in _pipelines:
            print(f'Loading {key}...')
            if use_controlnet:
                pipe, device = load_controlnet_pipeline(
                    sd_model_id=MODEL_IDS.get(model_key, MODEL_IDS['sd15'])
                )
            else:
                pipe, device = load_sd_pipeline(
                    model_id=MODEL_IDS.get(model_key, MODEL_IDS['sdxl']),
                    use_sdxl=(model_key == 'sdxl')
                )
            pipe = pipe.to(DTYPE)

            # torch.compile disabled — CUDA graphs break in Flask's thread pool
            print('  torch.compile skipped (Flask threading incompatible)')

            try:
                pipe.enable_xformers_memory_efficient_attention()
                print('  xformers memory-efficient attention enabled')
            except Exception:
                pipe.enable_attention_slicing()
                print('  attention slicing enabled')

            _pipelines[key] = (pipe, device)
            print(f'{key} ready')
    return _pipelines[key]

print('Pre-loading SDXL on A100 (60-90s)...')
get_pipeline('sdxl')
print('SDXL ready')


In [ ]:
# ── CELL 3 ── Flask API ──────────────────────────────────────────────────────
import base64, csv, io, time, traceback
from collections import defaultdict
from flask import Flask, request, jsonify
from flask_cors import CORS
from PIL import Image

app = Flask(__name__)
CORS(app)

QUALITY_FIELDNAMES = [
    'product_id',
    'product_title',
    'prompt_type',
    'image_name',
    'image_path',
    'quality_score',
    'quality_notes',
]

def pil_to_b64(img):
    buf = io.BytesIO()
    img.save(buf, format='PNG')
    return 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode()

def b64_to_pil(data_url):
    _, data = data_url.split(',', 1)
    return Image.open(io.BytesIO(base64.b64decode(data))).convert('RGB')

def run_inference(pipe, device, prompt, n, seed, cfg, steps, size, control_img=None):
    # A100 has 40GB — generate all N images in one batched call (much faster)
    generators = [torch.Generator(device=device).manual_seed(seed + i) for i in range(n)]
    kwargs = dict(
        prompt=[prompt] * n,
        negative_prompt=[NEGATIVE_PROMPT] * n,
        num_inference_steps=steps,
        guidance_scale=cfg,
        height=size, width=size,
        generator=generators,
    )
    if control_img is not None:
        kwargs['image'] = [control_img] * n
    else:
        kwargs['num_images_per_prompt'] = 1
    with torch.inference_mode():
        result = pipe(**kwargs)
    return result.images

def load_quality_rows():
    if not QUALITY_SCORES_PATH.exists():
        return {}
    with open(QUALITY_SCORES_PATH, 'r', encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        rows = {}
        for row in reader:
            image_name = row.get('image_name')
            if image_name:
                rows[image_name] = row
        return rows

def save_quality_rows(rows):
    with open(QUALITY_SCORES_PATH, 'w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=QUALITY_FIELDNAMES)
        writer.writeheader()
        writer.writerows(rows)

def write_quality_summary(rows):
    grouped = defaultdict(lambda: {'scores': [], 'product_title': ''})
    for row in rows:
        key = (row['product_id'], row['prompt_type'])
        try:
            score = float(row['quality_score'])
        except (TypeError, ValueError):
            continue
        grouped[key]['scores'].append(score)
        grouped[key]['product_title'] = row.get('product_title', '')

    summary_rows = []
    for (product_id, prompt_type), payload in sorted(grouped.items()):
        scores = payload['scores']
        if not scores:
            continue
        summary_rows.append({
            'product_id': product_id,
            'product_title': payload['product_title'],
            'prompt_type': prompt_type,
            'mean_quality_score': round(sum(scores) / len(scores), 2),
            'n_quality_scored': len(scores),
        })

    with open(QUALITY_SUMMARY_PATH, 'w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=['product_id', 'product_title', 'prompt_type', 'mean_quality_score', 'n_quality_scored']
        )
        writer.writeheader()
        writer.writerows(summary_rows)

    return summary_rows

@app.route('/health', methods=['GET'])
def health():
    free, total = torch.cuda.mem_get_info()
    return jsonify({
        'status'        : 'ok',
        'gpu'           : torch.cuda.get_device_name(0),
        'vram_total_gb' : round(total / 1e9, 1),
        'vram_free_gb'  : round(free  / 1e9, 1),
        'loaded_models' : list(_pipelines.keys()),
        'dtype'         : str(DTYPE),
        'quality_scores_path': str(QUALITY_SCORES_PATH),
        'quality_summary_path': str(QUALITY_SUMMARY_PATH),
    })

@app.route('/generate', methods=['POST'])
def generate():
    try:
        d = request.get_json(force=True)
        product = {
            'id'      : 'api',
            'title'   : d.get('title',    'Product'),
            'category': d.get('category', 'General'),
            'color'   : d.get('color',    ''),
            'material': d.get('material', ''),
            'style'   : d.get('style',    ''),
        }
        model_key = d.get('model', 'sdxl')
        defs      = DEFAULTS[model_key]
        size      = int(d.get('resolution', defs['resolution']))
        n         = min(int(d.get('n_images', 4)), 4)
        seed      = int(d.get('seed', 42))
        cfg       = float(d.get('cfg_scale', defs['cfg']))
        steps     = int(d.get('steps', defs['steps']))
        use_cn    = bool(d.get('use_controlnet', False))
        compare   = bool(d.get('compare_mode', True))
        custom_p  = d.get('custom_prompt', '').strip()

        s_prompt  = custom_p if custom_p else structured_prompt(product)
        n_prompt  = naive_prompt(product)

        control_img = None
        if use_cn and d.get('reference_image'):
            from pipeline.generator import build_canny_control_image
            import tempfile
            ref = b64_to_pil(d['reference_image']).resize((size, size))
            with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as tf:
                ref.save(tf.name)
                tmp_path = tf.name
            control_img = build_canny_control_image(tmp_path, target_size=(size, size))
            os.unlink(tmp_path)

        pipe, device = get_pipeline(model_key, use_controlnet=(use_cn and control_img is not None))

        t0 = time.time()
        struct_imgs = run_inference(pipe, device, s_prompt, n, seed,        cfg, steps, size, control_img)
        naive_imgs  = run_inference(pipe, device, n_prompt, n, seed + 1000, cfg, steps, size) if compare else []
        gen_time    = round(time.time() - t0, 1)

        clip_scores = {}
        try:
            clip_scores['structured'] = clip_score(struct_imgs[0], s_prompt)
            if naive_imgs:
                clip_scores['naive'] = clip_score(naive_imgs[0], n_prompt)
        except Exception as e:
            print(f'CLIP skipped: {e}')

        return jsonify({
            'structured_images': [pil_to_b64(i) for i in struct_imgs],
            'naive_images'     : [pil_to_b64(i) for i in naive_imgs],
            'structured_prompt': s_prompt,
            'naive_prompt'     : n_prompt,
            'clip_scores'      : clip_scores,
            'gen_time_s'       : gen_time,
            'model'            : model_key,
            'resolution'       : size,
        })

    except Exception:
        tb = traceback.format_exc()
        print(tb)
        return jsonify({'error': tb}), 500

@app.route('/quality/submit', methods=['POST'])
def submit_quality():
    try:
        d = request.get_json(force=True)
        image_name = d.get('image_name') or Path(d.get('image_path', '')).name
        quality_score = str(d.get('quality_score', '')).strip()

        if not image_name:
            return jsonify({'error': 'image_name or image_path is required'}), 400
        if quality_score not in {'1', '2', '3', '4', '5'}:
            return jsonify({'error': 'quality_score must be one of 1, 2, 3, 4, 5'}), 400

        row = {
            'product_id': d.get('product_id', 'api'),
            'product_title': d.get('product_title', ''),
            'prompt_type': d.get('prompt_type', 'structured'),
            'image_name': image_name,
            'image_path': d.get('image_path', ''),
            'quality_score': quality_score,
            'quality_notes': d.get('quality_notes', '').strip(),
        }

        rows = load_quality_rows()
        rows[image_name] = row
        ordered_rows = sorted(
            rows.values(),
            key=lambda item: (item['product_id'], item['prompt_type'], item['image_name'])
        )
        save_quality_rows(ordered_rows)
        summary_rows = write_quality_summary(ordered_rows)

        return jsonify({
            'status': 'saved',
            'saved_row': row,
            'total_scores': len(ordered_rows),
            'summary_rows': summary_rows,
            'quality_scores_path': str(QUALITY_SCORES_PATH),
            'quality_summary_path': str(QUALITY_SUMMARY_PATH),
        })

    except Exception:
        tb = traceback.format_exc()
        print(tb)
        return jsonify({'error': tb}), 500

@app.route('/quality/scores', methods=['GET'])
def get_quality_scores():
    rows = sorted(
        load_quality_rows().values(),
        key=lambda item: (item['product_id'], item['prompt_type'], item['image_name'])
    )
    return jsonify({
        'rows': rows,
        'count': len(rows),
        'quality_scores_path': str(QUALITY_SCORES_PATH),
    })

@app.route('/quality/summary', methods=['GET'])
def get_quality_summary():
    rows = []
    if QUALITY_SUMMARY_PATH.exists():
        with open(QUALITY_SUMMARY_PATH, 'r', encoding='utf-8', newline='') as handle:
            rows = list(csv.DictReader(handle))
    return jsonify({
        'rows': rows,
        'count': len(rows),
        'quality_summary_path': str(QUALITY_SUMMARY_PATH),
    })

print('Flask API ready  |  GET /health  |  POST /generate  |  POST /quality/submit')

In [ ]:
# ── CELL 4 ── Start ngrok + Flask  (keep this cell running\!) ─────────────────
from pyngrok import ngrok

PORT = 5000

# Optional: add your free ngrok token for longer sessions (no timeout)
# Get it at: https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token('NGROK_AUTH_TOKEN')  # <-- EDIT: add your ngrok token here or in the .env file
ngrok.kill()
tunnel = ngrok.connect(PORT, 'http')
url    = tunnel.public_url

print()
print('=' * 56)
print('  COLAB A100 API IS LIVE')
print('=' * 56)
print(f'  Public URL  : {url}')
print(f'  Health check: {url}/health')
print()
print('  Paste the Public URL into your HTML app')
print('  (Colab Backend box in the Live Demo tab)')
print()
print('  A100 defaults: SDXL · 1024px · 4 images · BF16')
print('  Batched generation: all 4 images in one forward pass')
print('=' * 56)
print('  Keep running. Runtime > Interrupt kernel to stop.')
print('=' * 56)
print()

app.run(port=PORT, use_reloader=False, debug=False, threaded=True)